<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/07-RAG_Improve_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Improving RAG with Metadata-Enriched Chunking

This notebook demonstrates how to improve a Retrieval-Augmented Generation (RAG) pipeline by enriching text chunks with LLM-generated metadata using LlamaIndex's metadata extractors.

## What You Will Learn
- How to apply `SummaryExtractor`, `QuestionsAnsweredExtractor`, and `KeywordExtractor` to enrich document chunks
- How to run an ingestion pipeline that embeds metadata-enriched nodes into a ChromaDB vector store
- How to evaluate retrieval quality using `RetrieverEvaluator` (hit rate, MRR) and `RelevancyEvaluator` / `FaithfulnessEvaluator`
- How metadata enrichment compares to a plain (no-metadata) baseline

## Prerequisites
- Familiarity with RAG concepts (chunking, embedding, vector stores)
- Basic knowledge of LlamaIndex pipelines
- An OpenAI API key (used for embeddings, the LLM, and evaluation)


# Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.19 openai==2.8.1 google-genai==1.70.0 llama-index-embeddings-openai==0.6.0 llama-index-llms-google-genai==0.9.0 \
                opentelemetry-api==1.38.0 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

# Load a Model


In [ ]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-5.4-mini", additional_kwargs={"reasoning_effort": "low"})

In [ ]:
# Gemini (Free Tier) - RPM Quota error might happen.

# from llama_index.llms.google_genai import GoogleGenAI
# llm = GoogleGenAI(model="gemini-2.5-flash", temperature=0.3)

# Create a Vector Store


In [ ]:
import chromadb

# create client and a new collection
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = chroma_client.get_or_create_collection("mini-llama-articles")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model. Read the dataset as a long string.


In [ ]:
!curl -o ./mini-llama-articles.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  169k  100  169k    0     0  1024k      0 --:--:-- --:--:-- --:--:-- 1027k


## Read File


In [ ]:
import csv

rows = []

# Load the CSV file
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        rows.append(row)

# The number of characters in the dataset.
len(rows)

14

# Convert to Document obj


In [ ]:
from llama_index.core import Document

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [
    Document(
        text=row[1], metadata={"title": row[0], "url": row[2], "source_name": row[3]}
    )
    for row in rows
]

# Chunk & Enrich with Metadata


In [ ]:
from llama_index.core.node_parser import TokenTextSplitter

# Define the splitter object that split the text into segments with 512 tokens,
# with a 128 overlap between the segments.
text_splitter = TokenTextSplitter(separator=" ", chunk_size=512, chunk_overlap=128)

In [ ]:
from llama_index.core.extractors import (
    SummaryExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

# Create the pipeline to apply the transformation on each chunk,
# and store the transformed text in the chroma vector store.
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        QuestionsAnsweredExtractor(questions=3, llm=llm),
        SummaryExtractor(summaries=["prev", "self"], llm=llm),
        KeywordExtractor(keywords=10, llm=llm),
        OpenAIEmbedding(),
    ],
    vector_store=vector_store,
)

# Run the transformation pipeline.
nodes = pipeline.run(documents=documents, show_progress=True)

Applying transformations:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 108/108 [00:41<00:00,  2.61it/s]

100%|██████████| 108/108 [01:07<00:00,  1.61it/s]

100%|██████████| 108/108 [00:28<00:00,  3.81it/s]


In [ ]:
len(nodes)

108

## Optional: Inspecting Extracted Metadata

In [ ]:
# Preview the metadata extracted by LlamaIndex for the first few nodes
print("=== Sample Extracted Metadata (first 3 nodes) ===\n")
for i, node in enumerate(nodes[:3]):
    print(f"Node {i+1}:")
    print(f"  Text preview : {node.text[:150]}...")
    for key, val in node.metadata.items():
        val_str = str(val)[:200] if val else "(empty)"
        print(f"  {key:<20}: {val_str}")
    print()

=== Sample Extracted Metadata (first 3 nodes) ===

Node 1:
  Text preview : LLM Variants and Meta's Open Source Before shedding light on four major trends, I'd share the latest Meta's Llama 2 and Code Llama. Meta's Llama 2 rep...
  title               : Beyond GPT-4: What's New?
  url                 : https://pub.towardsai.net/beyond-gpt-4-whats-new-cbd61a448eb9#dda8
  source_name         : towards_ai
  questions_this_excerpt_can_answer: Here are 3 questions this context can answer specifically:

1. **What parameter sizes are included in Meta’s Llama 2 model family?**  
2. **Which three variants of Code Llama are mentioned, and what i
  section_summary     : This section focuses on Meta’s open-source LLM family and the shift toward more capable models.

**Key topics:**
- **Llama 2** as Meta’s latest open-source model suite
- **Parameter sizes** spanning *
  excerpt_keywords    : Llama 2, Code Llama, Meta, open-source, GPT-4, LLMs, Python, Instruct, chat models, multimodal

Node 2:
  T

In [ ]:
# Compress the vector store directory to a zip file to be able to download and use later.
!zip -r vectorstore.zip mini-llama-articles

  adding: mini-llama-articles/ (stored 0%)
  adding: mini-llama-articles/665b7eab-8efa-4cce-a667-4b27d84616e2/ (stored 0%)
  adding: mini-llama-articles/665b7eab-8efa-4cce-a667-4b27d84616e2/link_lists.bin (stored 0%)
  adding: mini-llama-articles/665b7eab-8efa-4cce-a667-4b27d84616e2/header.bin (deflated 63%)
  adding: mini-llama-articles/665b7eab-8efa-4cce-a667-4b27d84616e2/length.bin (deflated 32%)
  adding: mini-llama-articles/665b7eab-8efa-4cce-a667-4b27d84616e2/data_level0.bin (deflated 100%)
  adding: mini-llama-articles/chroma.sqlite3 (deflated 68%)


# Load Indexes


If you have already uploaded the zip file for the vector store checkpoint, please uncomment the code in the following cell block to extract its contents. After doing so, you will be able to load the dataset from local storage.


In [ ]:
# !unzip vectorstore.zip

In [ ]:
# Load the vector store from the local storage.
db = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = db.get_or_create_collection("mini-llama-articles")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
from llama_index.core import VectorStoreIndex

# Create the index based on the vector store.
index = VectorStoreIndex.from_vector_store(vector_store)

# Query Dataset


In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

res = query_engine.query("How many parameters LLaMA2 model has?")

In [ ]:
print(res.response)

LLaMA 2 comes in four parameter sizes: **7B, 13B, 34B, and 70B**.


In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 2a8c034b-030e-49df-9e84-5cd9cc947040
Title	 Meta's Llama 2: Revolutionizing Open Source Language Models for Commercial Use
Text	 I. Llama 2: Revolutionizing Commercial Use Unlike its predecessor Llama 1, which was limited to research use, Llama 2 represents a major advancement as an open-source commercial model. Businesses can now integrate Llama 2 into products to create AI-powered applications. Availability on Azure and AWS facilitates fine-tuning and adoption. However, restrictions apply to prevent exploitation. Companies with over 700 million active daily users cannot use Llama 2. Additionally, its output cannot be used to improve other language models.  II. Llama 2 Model Flavors Llama 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters. While 7B, 13B, and 70B have already been released, the 34B model is still awaited. The pretrained variant, trained on a whopping 2 trillion tokens, boasts a context window of 4096 toke

### Trying a different Query


In [ ]:
res = query_engine.query("Does GQA helped LLaMA performance?")

In [ ]:
print(res.response)

That isn’t stated here, so I can’t confirm whether GQA improved LLaMA’s performance from the available information.


In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 00ca7c10-43f8-4253-8e26-670558798137
Title	 Exploring Large Language Models -Part 3
Text	 method to a proper QA dataset. These and other improvements need to be experimented with, as well as to train with some completely new data that the model has not seen to test more effectively. Update: Training with new data was done by writing an imaginary story with ChatGPT help and then creating an instruction tuning data set (colab notebook). The model was then trained and tested (colab notebook) with this generated instruct dataset. The results confirm that the model learns via Instruct tuning, not only the fed questions but other details and relations of the domain. Problems with hallucinations remain (Bordor, Lila characters who are not in the story). The LLama2 13B 4-bit fine-tuned model has better output than the 7B model. A lot more needs to be explored in Fine-tuning. One observation is that slight changes in prompts give different answers. Since the output is not deterministic

From the articles:

> [...]The 7 billion model of Llama2 has sufficient NLU (Natural Language Understanding) to create output based on a particular format[...]


# No Metadata


Now, let's evaluate the ability of the query engine independently of the generated metadata, like keyword extraction or summarization.


In [ ]:
from llama_index.core import Document

documents_no_meta = [Document(text=row[1]) for row in rows]

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        OpenAIEmbedding(),
    ]
)

nodes_no_meta = pipeline.run(documents=documents_no_meta, show_progress=True)

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
index_no_metadata = VectorStoreIndex(nodes=nodes_no_meta)

query_engine_no_metadata = index_no_metadata.as_query_engine(llm=llm)

In [ ]:
res = query_engine_no_metadata.query("Does GQA helped LLaMA performance?")

In [ ]:
print(res.response)

It isn’t stated here whether GQA improved LLaMA’s performance.


In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 487208eb-431c-4e8f-ba29-d237cfa1e984
Text	 137B parameters and perform instruction tuning ... Since we use QLoRa we are effectively closely following this paper - QLORA: Efficient Finetuning of Quantized LLMs concerning the training data set, the format that the authors used to train their Gauanco model This is the format for the Llama2 model and will be different for others. One of the hardest problems of training is finding or creating a good quality data set to train. In our case, converting the available training data set to the instruction data set. Since our use case is Closed Book QA, we need to convert this to a QA format. Using older NLP methods like NER (Named Entity Recognition) and then using that to create a QA dataset was not effective. This is where the Self-instruct concept could be used However previous to Llama2, the best-performing model was the GPT 3/4 model via ChatGPT or its API and using these models to do the same was expensive. The 7 billion model of L

# Evaluate


In [ ]:
from llama_index.core.evaluation import generate_question_context_pairs

# Create questions for each segment. These questions will be used to
# assess whether the retriever can accurately identify and return the
# corresponding segment when queried.

rag_eval_dataset = generate_question_context_pairs(
    nodes, llm=llm, num_questions_per_chunk=1
)

# We can save the evaluation dataset as a json file for later use.
rag_eval_dataset.save_json("./rag_eval_dataset.json")

100%|██████████| 108/108 [01:50<00:00,  1.02s/it]


If you have uploaded the generated question JSON file, please uncomment the code in the next cell block. This will avoid the need to generate the questions manually, saving you time and effort.


In [ ]:
# from llama_index.finetuning.embeddings.common import (
#     EmbeddingQAFinetuneDataset,
# )
# rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json(
#     "./rag_eval_dataset.json"
# )

In [ ]:
import pandas as pd


#  A simple function to show the evaluation result.
def display_results_retriever(name, eval_results):
    """Display results from evaluate."""

    metric_dicts = []
    for eval_result in eval_results:
        metric_dict = eval_result.metric_vals_dict
        metric_dicts.append(metric_dict)

    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    metric_df = pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

    return metric_df

In [ ]:
from llama_index.core.evaluation import RetrieverEvaluator

# We can evaluate the retrievers with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

    Retriever Name  Hit Rate       MRR
0  Retriever top_2  0.907407  0.837963
    Retriever Name  Hit Rate       MRR
0  Retriever top_4  0.972222  0.858796
    Retriever Name  Hit Rate       MRR
0  Retriever top_6  0.990741  0.862191
    Retriever Name  Hit Rate       MRR
0  Retriever top_8       1.0  0.863349
     Retriever Name  Hit Rate       MRR
0  Retriever top_10       1.0  0.863349


In [ ]:
from llama_index.core.evaluation import (
    RelevancyEvaluator,
    FaithfulnessEvaluator,
    BatchEvalRunner,
)
from llama_index.llms.openai import OpenAI

for i in [2, 4, 6, 8, 10]:
    # Set Faithfulness and Relevancy evaluators
    query_engine = index.as_query_engine(similarity_top_k=i, llm=llm)

    # While we use GPT-5-mini to answer questions, we can use GPT-5 to evaluate the answers.
    llm_gpt5 = OpenAI(model="gpt-5.4-mini", additional_kwargs={"reasoning_effort": "low"})

    faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_gpt5)
    relevancy_evaluator = RelevancyEvaluator(llm=llm_gpt5)

    # Run evaluation
    queries = list(rag_eval_dataset.queries.values())
    batch_eval_queries = queries[:20]

    runner = BatchEvalRunner(
        {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator},
        workers=8,
    )
    eval_results = await runner.aevaluate_queries(
        query_engine, queries=batch_eval_queries
    )
    faithfulness_score = sum(
        result.passing for result in eval_results["faithfulness"]
    ) / len(eval_results["faithfulness"])
    print(f"top_{i} faithfulness_score: {faithfulness_score}")

    relevancy_score = sum(result.passing for result in eval_results["relevancy"]) / len(
        eval_results["relevancy"]
    )
    print(f"top_{i} relevancy_score: {relevancy_score}")
    print("-_" * 10)

top_2 faithfulness_score: 1.0
top_2 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_4 faithfulness_score: 1.0
top_4 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_6 faithfulness_score: 1.0
top_6 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_8 faithfulness_score: 1.0
top_8 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_
top_10 faithfulness_score: 1.0
top_10 relevancy_score: 1.0
-_-_-_-_-_-_-_-_-_-_


## Optional: Token Chunk Size Effect on Retrieval

In [ ]:
# Demonstrate how chunk size affects the number of nodes produced
from llama_index.core.node_parser import TokenTextSplitter

test_sizes = [128, 256, 512, 1024]
print(f"{'Chunk size':>12} | {'Nodes produced':>15} | {'Avg tokens/node (approx)':>25}")
print("-" * 58)

for size in test_sizes:
    splitter = TokenTextSplitter(separator=" ", chunk_size=size, chunk_overlap=size // 4)
    test_nodes = splitter.get_nodes_from_documents(documents)
    avg_len = sum(len(n.text.split()) for n in test_nodes) / max(len(test_nodes), 1)
    print(f"{size:>12} | {len(test_nodes):>15} | {avg_len:>25.1f}")

print("\nNote: More nodes = finer granularity but higher embedding cost and latency.")

  Chunk size |  Nodes produced |  Avg tokens/node (approx)
----------------------------------------------------------
Metadata length (81) is close to chunk size (128). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
         128 |            1133 |                      47.4
         256 |             268 |                     144.7
         512 |             108 |                     329.8
        1024 |              50 |                     669.3

Note: More nodes = finer granularity but higher embedding cost and latency.
